## Task: Measure chirality level from the image
Measures of quantifying twists in a 2D surface. (possible use of Frenet-Serret equations)
- Haussdorf Distance
- Osipov-Pickup-Dunner index
    - Related to intrinsic molecular chirality tensor
- Continuous chirality measure

First we load the xlsx data

In [1]:
import pandas as pd
from pathlib import Path
file_name = 'Front_EE-1_1_3000x_rings_coords.csv'
# csv_file = Path('/Users/rishabhkumar/spiral-chirals/vf_exports/Front_EE-1_1_3000x_rings_coords.csv')
csv_file = Path.cwd() / 'vf_exports' / file_name
with open(csv_file, 'r') as f:
    df = pd.read_csv(f)

print(df.head())
# Try to find any .xlsx in the current working directory
# xlsx_files = list(Path('.').glob('*.xlsx'))

# if not xlsx_files:
#     raise FileNotFoundError(
#         "No .xlsx files found in the current directory. "
#         "Set `xlsx_path` to the file you want to load."
#     )

# # If multiple files found, pick the first one (adjust if you want a specific file)
# xlsx_path = xlsx_files[0]
# print(f"Loading Excel file: {xlsx_path}")

# # Read the first sheet into a dataframe
# df = pd.read_excel(xlsx_path, engine='openpyxl')

# # Quick inspection
# print("DataFrame shape:", df.shape)
# display(df.head())
# df.info()

FileNotFoundError: [Errno 2] No such file or directory: '/Users/rishabhkumar/spiral-chirals/python_notebooks/vf_exports/Front_EE-1_1_3000x_rings_coords.csv'

In [ ]:
# Extract angles and filter valid data
angles_deg = df[angle_col].values
X_plot = X[valid_mask]
Y_plot = Y[valid_mask]
angles_deg = angles_deg[valid_mask]

# Convert angles to radians and compute unit direction vectors
angles_rad = np.deg2rad(angles_deg)
U_plot = np.cos(angles_rad)
V_plot = np.sin(angles_rad)

# Compute a reasonable scale factor for the quiver key
scale_factor = np.round(np.percentile(np.sqrt(U_plot**2 + V_plot**2), 75), 1)
if scale_factor == 0:
    scale_factor = 1.0

# Create the plot
plt.figure(figsize=(9, 9))
magnitude_scale = 5.0
U_plot_scaled = magnitude_scale * U_plot
V_plot_scaled = magnitude_scale * V_plot

q = plt.quiver(
    X_plot, Y_plot, U_plot_scaled, V_plot_scaled, angles_deg,
    angles='xy', scale_units='xy', scale=1, cmap='hsv',
    width=0.08,
    headwidth=12,
    headlength=16,
    headaxislength=8,
    alpha=0.95, edgecolor='k', linewidth=0.3
)

plt.scatter(X_plot, Y_plot, c='k', s=20, zorder=3)

plt.gca().set_aspect('equal', 'box')
plt.xlabel('X')
plt.ylabel('Y')
plt.title(f'Vector field from coordinates and {angle_col}')

cbar = plt.colorbar(q, label=f'{angle_col} (deg)')
cbar.ax.tick_params(labelsize=9)

plt.quiverkey(q, X=0.92, Y=1.02, U=scale_factor,
              label=f'{scale_factor} data-units', labelpos='E',
              fontproperties={'size':10})

plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('vector_field_plot.svg', dpi=300)
plt.show()


NameError: name 'angle_col' is not defined

In [10]:
import numpy as np
import pandas as pd

# Import the dataset from the CSV file
df = pd.read_csv('sample_dataset.csv')

# Extract X and Y coordinates from dataframe
# Try to extract from Coordinate column if it exists, otherwise from X and Y columns
if 'Coordinate' in df.columns:
    coordinates = df['Coordinate'].str.strip('()').str.split(',', expand=True).astype(float)
    X = coordinates[0].values
    Y = coordinates[1].values
else:
    X = df['X'].values.astype(float)
    Y = df['Y'].values.astype(float)

# Define the angle column name
angle_col = 'Angle (α′)'

# Filter out any invalid data points
valid_mask = ~(np.isnan(X) | np.isnan(Y))

# Calculate dataset-adaptive bandwidth parameters
s_X = max(np.ptp(X), np.ptp(Y), 1)  # ptp = peak-to-peak (range)
d_nn = np.median(np.sort(np.sqrt(np.diff(X)**2 + np.diff(Y)**2))[:10])  # approximate median NN spacing

# Define bandwidth grid endpoints
ell_min = max(d_nn/2, s_X/80, 1e-3)
ell_max = max(1.5*s_X, 2*d_nn, 4*s_X/80)

# Create geometric spacing with 14 points
space = np.geomspace(ell_min, ell_max, 14)
print(space)

[ 16.34778272  22.00168519  29.61099737  39.85200031  53.63486779
  72.18455838  97.1496842  130.74903209 175.96875928 236.82778945
 318.73499641 428.96992016 577.32973938 777.        ]
